In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path('..').resolve()
FISHER_TAB = ROOT / 'results' / 'fisher_day0_day15' / 'tables'
OUT = ROOT / 'coursework_submission_notebooks' / 'outputs' / '05_clustering'
TAB = OUT / 'tables'
FIG = OUT / 'figures'
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
DONORS = ['P1', 'P2', 'Q1', 'Q2', 'S1', 'S2']
AMINO = list('ACDEFGHIKLMNPQRSTVWY')
try:
    import logomaker
    HAS_LOGO = True
except Exception:
    HAS_LOGO = False


In [ ]:
parts = []
for d in DONORS:
    p = pd.read_csv(FISHER_TAB / f'fisher_positive_FC4_{d}.csv')
    parts.append(p)
markers = pd.concat(parts, ignore_index=True)
markers.head()


In [ ]:
class UnionFind:

    def __init__(self, items):
        self.p = {x: x for x in items}

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        (ra, rb) = (self.find(a), self.find(b))
        if ra != rb:
            self.p[rb] = ra

def hamming_le1(a, b):
    if len(a) != len(b):
        return False
    diff = 0
    for (x, y) in zip(a, b):
        if x != y:
            diff += 1
            if diff > 1:
                return False
    return True

def build_components(seqs):
    by_len = defaultdict(list)
    for s in seqs:
        by_len[len(s)].append(s)
    aa_to_root = {}
    edges = []
    for (L, group) in by_len.items():
        uniq = list(dict.fromkeys(group))
        uf = UnionFind(uniq)
        Sset = set(uniq)
        for s in uniq:
            chars = list(s)
            for pos in range(L):
                orig = chars[pos]
                for alt in AMINO:
                    if alt == orig:
                        continue
                    chars[pos] = alt
                    mut = ''.join(chars)
                    if mut in Sset and mut > s:
                        uf.union(s, mut)
                        edges.append((s, mut))
                chars[pos] = orig
        for s in uniq:
            aa_to_root[s] = f'L{L}_{uf.find(s)}'
    return (edges, aa_to_root)
unique_aa = sorted(set(markers['cdr3_aa'].astype(str)))
(edges, aa_to_root) = build_components(unique_aa)
root_to_members = defaultdict(list)
for (aa, root) in aa_to_root.items():
    root_to_members[root].append(aa)
cluster_id = {}
summaries = []
cid = 0
for (root, members) in sorted(root_to_members.items(), key=lambda x: -len(x[1])):
    cid += 1
    name = f'C{cid:04d}'
    for aa in members:
        cluster_id[aa] = name
    summaries.append({'cluster_id': name, 'size': len(members), 'cdr3_length': len(members[0]), 'members': ';'.join(members)})
cluster_summary = pd.DataFrame(summaries)
markers = markers.copy()
markers['cluster_id'] = markers['cdr3_aa'].map(cluster_id)
markers.to_csv(TAB / 'markers_with_clusters.csv', index=False)
cluster_summary.to_csv(TAB / 'cluster_summary.csv', index=False)
n_multi = int((cluster_summary['size'] >= 2).sum())
n_in_multi = int(cluster_summary.loc[cluster_summary['size'] >= 2, 'size'].sum())
cluster_summary.head(10)


In [ ]:
sizes = cluster_summary['size'].value_counts().sort_index()
(fig, ax) = plt.subplots(figsize=(8, 4))
ax.bar(sizes.index.astype(str), sizes.values, color='#4c72b0')

plt.tight_layout()
plt.savefig(FIG / 'cluster_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
top = cluster_summary.head(10)
if HAS_LOGO:
    n = len(top)
    (fig, axes) = plt.subplots(n, 1, figsize=(10, 1.4 * n))
    if n == 1:
        axes = [axes]
    for (ax, row) in zip(axes, top.itertuples()):
        seqs = row.members.split(';')
        counts = logomaker.alignment_to_matrix(seqs, to_type='counts')
        info = logomaker.transform_matrix(counts, from_type='counts', to_type='information')
        logomaker.Logo(info, ax=ax, color_scheme='chemistry', show_spines=False)

    plt.tight_layout()
    plt.savefig(FIG / 'top10_sequence_logos.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(top[['cluster_id', 'size', 'cdr3_length']])
